# Titanic 生还预测

整理自最新上传的 notebook9d78c9203f.ipynb。保留原实验逻辑，使用 5 个特征；旧版 9 特征实验可从 Git 历史查看。运行前在 Kaggle 添加 Titanic 竞赛数据，无需 GPU。

## 1. 导入依赖

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier


## 2. 加载数据

In [ ]:
## 数据导入
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")


## 3. 填充缺失值

In [ ]:
## 缺失值处理
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())
train_data['Fare'] = train_data['Fare'].fillna(train_data['Fare'].median())
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())


## 4. 年龄分箱

In [ ]:
## 分箱
bins = [0, 12, 18, 60, 100]
labels = [0, 1, 2, 3]
train_data['Age_Bin'] = pd.cut(train_data['Age'], bins=bins, labels=labels)
test_data['Age_Bin'] = pd.cut(test_data['Age'], bins=bins, labels=labels)


## 5. 家庭特征

In [ ]:
## 家庭大小 
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)


## 6. 编码与列对齐

In [ ]:
##特征提取
features = ["Pclass", "Sex", "Age_Bin", "FamilySize","IsAlone"]
##维度轴数+维度对齐
x = pd.get_dummies(train_data[features])
y = train_data["Survived"]
x_test = pd.get_dummies(test_data[features])
x_test = x_test.reindex(columns=x.columns, fill_value=0)


## 7. 训练与预测

In [ ]:
##模型设置
model = RandomForestClassifier(n_estimators=100,random_state=91)
model.fit(x,y)
predictions = model.predict(x_test)


## 8. 生成提交文件

In [ ]:
##输出
submission = pd.DataFrame({"PassengerId":test_data["PassengerId"],"Survived":predictions})
submission.to_csv("submission.csv", index=False)


## 复盘与下一步

- 当前分别使用训练集和测试集自身的中位数填充。后续应统一使用训练部分统计量；Fare 当前没有进入模型。
- 年龄边界 [0, 12, 18, 60, 100] 是自定义不等宽分箱，默认左开右闭，0 或超出边界的年龄会成为缺失值。
- 当前未进行本地验证。下一步先划分训练和验证数据，再仅用训练部分拟合预处理规则，记录基线和改进版本的验证 Accuracy。
- 固定随机种子有助于复现，但不能证明历史分数变化全部来自随机种子。
- 历史 README 记录 0.77751（V13），尚无提交记录将其与某份代码明确对应。